# 31 Gdrive uploading - downloading

Uploads a text file to a shared folder using a personal Google Account.

If 'token.json' does not exist, it should pop up a Google login/consent window the first time since it's a fresh OAuth client.

Create an OAuth Client ID (not a service account) in Google Cloud Console → APIs & Services → Credentials → "Create Credentials" → "OAuth client ID" → Application type: Desktop app. Download it as e.g. secrets/oauth_client.json.

Consent screen needs this:
- Go to the Google Cloud Console.
- Select your project (Mapa).
- In the left menu, go to APIs & Services → OAuth consent screen (Pantalla de consentimiento de OAuth).
- Scroll down to the Test users (Usuarios de prueba) section. (Audience)
- Click + ADD USERS and enter your personal Gmail address.
- Click Save. (Check that the user was added to the list, this last save is tricky)

In [ ]:
from pathlib import Path

articles_dir = Path.cwd() / ".." / "data" / "articles"

In [ ]:
text = 'Hola Google'
filename = 'koko.txt'

article_path = articles_dir / filename

# Save to txt
with open(article_path, "w", encoding="utf-8") as f:
    f.write(text)

In [ ]:
import re

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# drive.file scope: app can only see/manage files it created or that the user opens with it
# SCOPES = ["https://www.googleapis.com/auth/drive.file"]
# Full drive scope: needed to read/download files other users uploaded to the shared folder
SCOPES = ["https://www.googleapis.com/auth/drive"]

oauth_client_path = Path.cwd() / ".." / "secrets" / "oauth_client.json"
token_path = Path.cwd() / ".." / "secrets" / "token.json"

url_folder = "https://drive.google.com/drive/folders/1i5Jl-Oho9k6PJPk4apsaUTwZbTCh8-oV?usp=sharing"
folder_id = re.search(r"/folders/([^/?]+)", url_folder).group(1)

# Reuse a cached token across runs; only pops up the browser login when needed
creds = None
if token_path.exists():
    creds = Credentials.from_authorized_user_file(str(token_path), SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(str(oauth_client_path), SCOPES)
        creds = flow.run_local_server(port=0)
    token_path.write_text(creds.to_json())

drive_service = build("drive", "v3", credentials=creds)

file_metadata = {"name": article_path.name, "parents": [folder_id]}
media = MediaFileUpload(str(article_path), mimetype="text/plain")
uploaded_file = drive_service.files().create(
    body=file_metadata, media_body=media, fields="id, name"
).execute()

print(f"Uploaded '{uploaded_file['name']}' with id: {uploaded_file['id']}")


## Downloading

Not tested yet.

In [ ]:
import io
from googleapiclient.http import MediaIoBaseDownload

def list_files_in_folder(folder_id):
    results = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed = false",
        fields="files(id, name)",
    ).execute()
    return results.get("files", [])

def download_file_from_drive(file_id, destination_path):
    request = drive_service.files().get_media(fileId=file_id)
    with io.FileIO(destination_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

# Example: download koko.txt back from the shared folder
files = list_files_in_folder(folder_id)
target = next(f for f in files if f["name"] == "koko.txt")
download_file_from_drive(target["id"], articles_dir / "koko_downloaded.txt")